<a href="https://colab.research.google.com/github/Heng1222/VeriPromiseESG_2026_TEAM_9906/blob/feat-model-train/app/model/model_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install transformers torch pandas numpy scikit-learn tqdm

In [4]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup, BertTokenizerFast
from sklearn.metrics import f1_score
import datetime

def load_fold_data_from_github(fold_num):
    """
    從 GitHub Raw 連結動態讀取指定 Fold 的訓練與驗證資料。
    :param fold_num: 整數 (1 到 5)
    :return: train_df, val_df
    """
    # 轉換為 GitHub Raw URL 格式
    base_url = "https://raw.githubusercontent.com/Heng1222/VeriPromiseESG_2026_TEAM_9906/main/app/data/clean_data/"
    train_url = f"{base_url}train_fold_{fold_num}.csv"
    val_url = f"{base_url}val_fold_{fold_num}.csv"

    print(f"\n📥 正在從 GitHub 獲取 Fold {fold_num} 的資料...")
    try:
        train_df = pd.read_csv(train_url)
        val_df = pd.read_csv(val_url)
        print(f"✅ 獲取成功！訓練集: {len(train_df)} 筆 | 驗證集: {len(val_df)} 筆")
        return train_df, val_df
    except Exception as e:
        print(f"❌ 讀取失敗，請檢查儲存庫是否設為 Public 或網址是否正確。錯誤訊息: {e}")
        return None, None

# ==========================================
# 0. 全域設定與超參數
# ==========================================
MODEL_NAME = "ckiplab/bert-base-chinese"
MAX_LEN = 512
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-5
OUTPUT_DIR = "./folds_data/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ==========================================
# 1. 資料前處理與 Dataset (包含 Head+Tail 截斷)
# ==========================================
class ESG_MTL_Dataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512, is_test=False):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

        self.head_len = int((max_len - 2) * 0.25)
        self.tail_len = (max_len - 2) - self.head_len

        # 標籤映射
        self.t1_map = {"No": 0, "Yes": 1}
        self.t2_map = {"already": 0, "within_2_years": 1, "between_2_and_5_years": 2, "longer_than_5_years": 3, "more_than_5_years": 3}
        self.t3_map = {"No": 0, "Yes": 1}
        self.t4_map = {"Clear": 0, "Not Clear": 1, "Misleading": 2}

    def _process_esg_type(self, esg_val):
        if pd.isna(esg_val) or str(esg_val).strip() == "": return "[ESG_UNK]"
        return " ".join([f"[ESG_{t.strip()}]" for t in str(esg_val).split(';')])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. 文本拼接
        esg_prefix = self._process_esg_type(row.get('esg_type', ''))
        raw_text = str(row['data'])
        full_text = f"{esg_prefix} 文本內容：{raw_text}"

        # 2. Token-level 雙向截斷
        tokens = self.tokenizer.encode(full_text, add_special_tokens=False)
        if len(tokens) > (self.max_len - 2):
            tokens = tokens[:self.head_len] + tokens[-self.tail_len:]

        input_ids = [self.tokenizer.cls_token_id] + tokens + [self.tokenizer.sep_token_id]
        attention_mask = [1] * len(input_ids)

        # 3. Padding
        pad_len = self.max_len - len(input_ids)
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        attention_mask += [0] * pad_len

        item = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long)
        }

        # 推論模式不回傳 Labels
        if self.is_test: return item

        # 4. 標籤轉換 (若為 N/A 或缺失，設為 -1 供 Loss 函數忽略)
        item['t1_label'] = torch.tensor(self.t1_map.get(str(row.get('promise_status')), -1), dtype=torch.float)
        item['t2_label'] = torch.tensor(self.t2_map.get(str(row.get('verification_timeline')), -1), dtype=torch.long)
        item['t3_label'] = torch.tensor(self.t3_map.get(str(row.get('evidence_status')), -1), dtype=torch.float)
        item['t4_label'] = torch.tensor(self.t4_map.get(str(row.get('evidence_quality')), -1), dtype=torch.long)

        return item

# ==========================================
# 2. 統一多任務模型架構 (MTL Backbone + 4 Heads)
# ==========================================
class ESG_Unified_MTL_Model(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size

        # T1 & T3 (二元分類): Multi-Sample Dropout 防禦過擬合
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])
        self.t1_head = nn.Linear(hidden_size, 1)
        self.t3_head = nn.Linear(hidden_size, 1)

        # T2 (4分類) & T4 (3分類)
        self.t2_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden_size, 4))
        self.t4_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden_size, 3))

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :] # 提取 [CLS] 向量

        # Multi-Sample Dropout 平均化
        t1_logits = torch.mean(torch.stack([self.t1_head(d(cls_output)) for d in self.dropouts]), dim=0).squeeze(-1)
        t3_logits = torch.mean(torch.stack([self.t3_head(d(cls_output)) for d in self.dropouts]), dim=0).squeeze(-1)

        t2_logits = self.t2_head(cls_output)
        t4_logits = self.t4_head(cls_output)

        return t1_logits, t2_logits, t3_logits, t4_logits

# ==========================================
# 3. 損失函數與屏蔽機制 (Masked Loss)
# ==========================================
def calculate_mtl_loss(preds, labels):
    t1_pred, t2_pred, t3_pred, t4_pred = preds
    t1_lbl, t2_lbl, t3_lbl, t4_lbl = labels

    # T1 損失 (二元)
    bce_loss = nn.BCEWithLogitsLoss()
    loss_t1 = bce_loss(t1_pred, t1_lbl)

    # 動態遮罩：找出 T1 真實為 Yes 的樣本，T2~T4 只對這些樣本計算 Loss
    valid_mask = (t1_lbl == 1)

    loss_t2 = loss_t3 = loss_t4 = torch.tensor(0.0, device=DEVICE)

    if valid_mask.sum() > 0:
        # T2 損失 (多分類，忽略 -1)
        ce_loss_t2 = nn.CrossEntropyLoss(ignore_index=-1)
        loss_t2 = ce_loss_t2(t2_pred[valid_mask], t2_lbl[valid_mask])

        # T3 損失 (二元，手動過濾 -1)
        t3_valid = valid_mask & (t3_lbl != -1)
        if t3_valid.sum() > 0:
            loss_t3 = bce_loss(t3_pred[t3_valid], t3_lbl[t3_valid])

        # T4 損失 (多分類，針對 Misleading 加權)
        t4_valid = valid_mask & (t4_lbl != -1)
        if t4_valid.sum() > 0:
            # 第一性原理：對付極端不平衡，給 Clear(0) 較低權重，給 Misleading(2) 極高權重
            t4_weights = torch.tensor([1.0, 2.0, 5.0], device=DEVICE)
            ce_loss_t4 = nn.CrossEntropyLoss(weight=t4_weights, ignore_index=-1)
            loss_t4 = ce_loss_t4(t4_pred[t4_valid], t4_lbl[t4_valid])

    # 任務權重調配 (基於競賽配分)
    total_loss = 0.2 * loss_t1 + 0.15 * loss_t2 + 0.3 * loss_t3 + 0.35 * loss_t4
    return total_loss

# ==========================================
# 4. 訓練流程與推論管線
# ==========================================
def train_and_predict():
    print("🚀 初始化 Tokenizer...")
    # tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer = BertTokenizerFast.from_pretrained('bert-base-chinese')
    special_tokens = {'additional_special_tokens': ['[ESG_E]', '[ESG_S]', '[ESG_G]', '[ESG_UNK]']}
    tokenizer.add_special_tokens(special_tokens)

    # K-Fold 訓練迴圈
    for fold in range(1, 6): # 執行 Fold 1 到 Fold 5
      print(f"\n{'='*40}")
      print(f"🏆 開始訓練 Fold {fold}")
      print(f"{'='*40}")

      # 1. 呼叫你的 GitHub 讀取函數
      train_df, val_df = load_fold_data_from_github(fold)
      if train_df is None or val_df is None:
          continue # 若讀取失敗則跳過該折

      # 2. 準備 Dataset 與 DataLoader
      train_dataset = ESG_MTL_Dataset(train_df, tokenizer, MAX_LEN)
      train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

      # 3. 實例化全新的模型 (確保每個 Fold 的權重都是重新初始化的)
      model = ESG_Unified_MTL_Model(MODEL_NAME)
      model.backbone.resize_token_embeddings(len(tokenizer))
      model.to(DEVICE)

      optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

      # 4. 開始該 Fold 的 Epoch 訓練
      model.train()
      for epoch in range(EPOCHS):
          total_loss = 0
          for batch in train_loader:
              optimizer.zero_grad()

              input_ids = batch['input_ids'].to(DEVICE)
              attention_mask = batch['attention_mask'].to(DEVICE)
              labels = (batch['t1_label'].to(DEVICE), batch['t2_label'].to(DEVICE),
                        batch['t3_label'].to(DEVICE), batch['t4_label'].to(DEVICE))

              preds = model(input_ids, attention_mask)
              loss = calculate_mtl_loss(preds, labels)

              loss.backward()
              optimizer.step()
              total_loss += loss.item()

          print(f"  - Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss/len(train_loader):.4f}")

      # 5. 儲存該 Fold 的專屬模型權重
      save_path = f"best_mtl_model_fold_{fold}.pth"
      torch.save(model.state_dict(), save_path)
      print(f"💾 Fold {fold} 模型已儲存至 {save_path}")
      # --- 最終目標：推論未來新資料並輸出 CSV ---
      print("\n🔮 準備執行新資料推論...")

      # 模擬一筆官方發布的未來新資料 (測試集)
      test_data = {
          'id': [99991, 99992],
          'esg_type': ['E', 'S;G'],
          'data': ['本公司承諾於 2030 年達成 100% 綠電使用，目前已導入太陽能版，預計每年減碳 10%。',
                  '我們致力於促進員工福祉，但具體計畫還在研議中。']
      }
      test_df = pd.DataFrame(test_data)
      test_dataset = ESG_MTL_Dataset(test_df, tokenizer, MAX_LEN, is_test=True)
      test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

      model.eval()
      results = []

      # 反向映射字典
      inv_t1 = {0: "No", 1: "Yes"}
      inv_t2 = {0: "already", 1: "within_2_years", 2: "between_2_and_5_years", 3: "longer_than_5_years"}
      inv_t3 = {0: "No", 1: "Yes"}
      inv_t4 = {0: "Clear", 1: "Not Clear", 2: "Misleading"}

      with torch.no_grad():
          for batch in test_loader:
              input_ids = batch['input_ids'].to(DEVICE)
              attention_mask = batch['attention_mask'].to(DEVICE)

              t1_log, t2_log, t3_log, t4_log = model(input_ids, attention_mask)

              # 轉換為類別預測
              t1_preds = (torch.sigmoid(t1_log) > 0.5).long().cpu().numpy()
              t2_preds = torch.argmax(t2_log, dim=1).cpu().numpy()
              t3_preds = (torch.sigmoid(t3_log) > 0.5).long().cpu().numpy()
              t4_preds = torch.argmax(t4_log, dim=1).cpu().numpy()

              # 路由邏輯 (Inference Routing)
              for i in range(len(t1_preds)):
                  if t1_preds[i] == 0:
                      results.append({"promise_status": "No", "verification_timeline": "N/A", "evidence_status": "N/A", "evidence_quality": "N/A"})
                  else:
                      t2_res = inv_t2[t2_preds[i]]
                      if t3_preds[i] == 0:
                          results.append({"promise_status": "Yes", "verification_timeline": t2_res, "evidence_status": "No", "evidence_quality": "N/A"})
                      else:
                          results.append({"promise_status": "Yes", "verification_timeline": t2_res, "evidence_status": "Yes", "evidence_quality": inv_t4[t4_preds[i]]})

      # 合併 ID 並輸出
      output_df = pd.DataFrame({'id': test_df['id']})
      pred_df = pd.DataFrame(results)
      final_output = pd.concat([output_df, pred_df], axis=1)

      # timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
      # output_filename = f"output_{timestamp}.csv"
      # final_output.to_csv(output_filename, index=False)
      # print(f"🎉 推論完成！結果已儲存至 {output_filename}")
      print(final_output)
      # 為了展示效率，目前先用 del 清除記憶體，避免 CUDA OOM
      del model
      torch.cuda.empty_cache()
    print("\n🎉 5-Fold 訓練全數完成！你現在擁有了 5 個不同視角的專家模型。")


if __name__ == "__main__":
    train_and_predict()

🚀 初始化 Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertModel LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstr

  - Epoch 1/5 | Train Loss: 0.6285
  - Epoch 2/5 | Train Loss: 0.4204
  - Epoch 3/5 | Train Loss: 0.3507
  - Epoch 4/5 | Train Loss: 0.2698
  - Epoch 5/5 | Train Loss: 0.1768
💾 Fold 4 模型已儲存至 best_mtl_model_fold_4.pth

🔮 準備執行新資料推論...
      id promise_status  verification_timeline evidence_status  \
0  99991            Yes    longer_than_5_years             Yes   
1  99992            Yes  between_2_and_5_years              No   

  evidence_quality  
0        Not Clear  
1              N/A  

🏆 開始訓練 Fold 5

📥 正在從 GitHub 獲取 Fold 5 的資料...
✅ 獲取成功！訓練集: 911 筆 | 驗證集: 200 筆


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstr

  - Epoch 1/5 | Train Loss: 0.6583
  - Epoch 2/5 | Train Loss: 0.4529
  - Epoch 3/5 | Train Loss: 0.3715
  - Epoch 4/5 | Train Loss: 0.2926
  - Epoch 5/5 | Train Loss: 0.2080
💾 Fold 5 模型已儲存至 best_mtl_model_fold_5.pth

🔮 準備執行新資料推論...
      id promise_status  verification_timeline evidence_status  \
0  99991            Yes    longer_than_5_years              No   
1  99992            Yes  between_2_and_5_years              No   

  evidence_quality  
0              N/A  
1              N/A  

🎉 5-Fold 訓練全數完成！你現在擁有了 5 個不同視角的專家模型。


In [5]:
# import pandas as pd
# import numpy as np
# import torch
# from torch.utils.data import DataLoader
# from tqdm import tqdm
# import gc

# def ensemble_inference_and_export(test_csv_path, output_csv_path="output.csv"):
#     print("🚀 啟動 Soft-Voting 記憶體置換集成推論管線...")

#     # 1. 讀取測試資料與 Dataset 初始化
#     # 假設測試集有 'id', 'data', 'esg_type' 欄位
#     test_df = pd.read_csv(test_csv_path)

#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     special_tokens = {'additional_special_tokens': ['[ESG_E]', '[ESG_S]', '[ESG_G]', '[ESG_UNK]']}
#     tokenizer.add_special_tokens(special_tokens)

#     # 注意：is_test=True 讓 Dataset 不會去尋找 label 欄位
#     test_dataset = ESG_MTL_Dataset(test_df, tokenizer, MAX_LEN, is_test=True)
#     test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#     num_samples = len(test_df)

#     # 2. 初始化 Logits 累加器 (Numpy Arrays 存於 CPU RAM，節省 VRAM)
#     acc_t1_logits = np.zeros(num_samples)
#     acc_t2_logits = np.zeros((num_samples, 4))
#     acc_t3_logits = np.zeros(num_samples)
#     acc_t4_logits = np.zeros((num_samples, 3))

#     # 3. 記憶體置換迴圈：逐一載入 5 個 Fold 的模型
#     for fold in range(1, 6):
#         print(f"\n📂 載入 Fold {fold} 專家權重...")
#         model = ESG_Unified_MTL_Model(MODEL_NAME)
#         model.backbone.resize_token_embeddings(len(tokenizer))

#         # 載入我們在前一階段訓練好的權重
#         weight_path = f"best_mtl_model_fold_{fold}.pth"
#         model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
#         model.to(DEVICE)
#         model.eval()

#         fold_t1, fold_t2, fold_t3, fold_t4 = [], [], [], []

#         with torch.no_grad():
#             for batch in tqdm(test_loader, desc=f"Fold {fold} 推論中"):
#                 input_ids = batch['input_ids'].to(DEVICE)
#                 attention_mask = batch['attention_mask'].to(DEVICE)

#                 # 取得該批次的 Logits
#                 t1_log, t2_log, t3_log, t4_log = model(input_ids, attention_mask)

#                 fold_t1.append(t1_log.cpu().numpy())
#                 fold_t2.append(t2_log.cpu().numpy())
#                 fold_t3.append(t3_log.cpu().numpy())
#                 fold_t4.append(t4_log.cpu().numpy())

#         # 將批次結果拼接並累加到全域累加器中
#         acc_t1_logits += np.concatenate(fold_t1, axis=0)
#         acc_t2_logits += np.concatenate(fold_t2, axis=0)
#         acc_t3_logits += np.concatenate(fold_t3, axis=0)
#         acc_t4_logits += np.concatenate(fold_t4, axis=0)

#         # 核心防禦：強制釋放 VRAM，防止 CUDA OOM
#         del model
#         gc.collect()
#         torch.cuda.empty_cache()

#     # 4. 軟投票平均 (Soft-Voting Averaging)
#     print("\n🧠 執行 Logits 平均與機率映射...")
#     avg_t1_logits = acc_t1_logits / 5.0
#     avg_t2_logits = acc_t2_logits / 5.0
#     avg_t3_logits = acc_t3_logits / 5.0
#     avg_t4_logits = acc_t4_logits / 5.0

#     # 5. 激勵函數與最終決策邊界 (Decision Boundary)
#     # T1 & T3 轉 Sigmoid 機率；T2 & T4 找最大機率的索引
#     # 這裡預設二元分類的閥值(Threshold)為 0.5，後續可透過驗證集最佳化
#     t1_preds = (1 / (1 + np.exp(-avg_t1_logits)) > 0.5).astype(int)
#     t3_preds = (1 / (1 + np.exp(-avg_t3_logits)) > 0.5).astype(int)
#     t2_preds = np.argmax(avg_t2_logits, axis=1)
#     t4_preds = np.argmax(avg_t4_logits, axis=1)

#     # 6. 逆向對齊字典 (Inverse Mapping)
#     inv_t1 = {0: "No", 1: "Yes"}
#     inv_t2 = {0: "already", 1: "within_2_years", 2: "between_2_and_5_years", 3: "longer_than_5_years"}
#     inv_t3 = {0: "No", 1: "Yes"}
#     inv_t4 = {0: "Clear", 1: "Not Clear", 2: "Misleading"}

#     # 7. 嚴格路由管線 (Strict Routing Pipeline)
#     # 第一性原理：如果上游預測為 No，下游必須強制為 N/A，否則會被競賽平台判定為格式錯誤
#     results = []
#     for i in range(num_samples):
#         if t1_preds[i] == 0:
#             # 任務一判定無承諾，後續全數截斷
#             results.append({
#                 "id": test_df.iloc[i]['id'],
#                 "promise_status": "No",
#                 "verification_timeline": "N/A",
#                 "evidence_status": "N/A",
#                 "evidence_quality": "N/A"
#             })
#         else:
#             # 任務一有承諾，繼續解析時間軸與證據
#             t2_res = inv_t2[t2_preds[i]]
#             if t3_preds[i] == 0:
#                 results.append({
#                     "id": test_df.iloc[i]['id'],
#                     "promise_status": "Yes",
#                     "verification_timeline": t2_res,
#                     "evidence_status": "No",
#                     "evidence_quality": "N/A"
#                 })
#             else:
#                 results.append({
#                     "id": test_df.iloc[i]['id'],
#                     "promise_status": "Yes",
#                     "verification_timeline": t2_res,
#                     "evidence_status": "Yes",
#                     "evidence_quality": inv_t4[t4_preds[i]]
#                 })

#     # 8. 輸出最終結果
#     final_output = pd.DataFrame(results)
#     final_output.to_csv(output_csv_path, index=False)
#     print(f"🎉 推論完成！高精度預測結果已匯出至: {output_csv_path}")
#     print(final_output.head())

# # ==========================================
# # 啟動指令範例：
# # ensemble_inference_and_export("你的測試集路徑.csv", "final_submission.csv")
# # ==========================================